## 1. Setup and Imports

In [ ]:
try:
    import torch_dct
except ImportError:
    !pip install torch_dct
    !pip install huffman
    !apt-get update && apt-get install -y ffmpeg

In [ ]:
# Add project root to path
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))
print(f"Added {project_root} to Python path")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader
from pathlib import Path
import lovely_tensors as lt
from tqdm.auto import tqdm

# Import our dataset classes
from sdate.datasets.projection_triplet_dataset import (
    ProjectionTripletDataset,
    TomographyFolderProcessor,
    create_train_val_split,
    load_tomography_params
)

# Set up lovely_tensors for better tensor visualization
lt.monkey_patch()

# Set up matplotlib
plt.rcParams['figure.figsize'] = (14, 10)
plt.rcParams['image.cmap'] = 'gray'

print("✅ Libraries imported successfully!")

## 2. Configure Data Paths

Set the path(s) to your tomographic data folders. Each folder should contain:
- Dark field images (first `num_darks` files)
- Flat field images (next `num_flats` files)
- Projection images (remaining files)

In [ ]:
# Configuration
# =============

# Multiple folder example
DATA_PATH = [Path(f'/myhome/data/sdate/shared/compression_paper/file_{i}_extracted') for i in range(1, 13)]

# Or single folder (uncomment and modify as needed)
# DATA_PATH = Path('/myhome/data/sdate/shared/compression_paper/file_9_extracted')

# Processing parameters
TARGET_SIZE = 512  # Standardized output size

# Check if paths exist and auto-detect tomography parameters
if isinstance(DATA_PATH, list):
    # Multiple folders
    valid_paths = [p for p in DATA_PATH if p.exists()]
    print(f"✅ Found {len(valid_paths)} valid folders out of {len(DATA_PATH)}")
    
    for path in valid_paths[:3]:  # Show first 3
        tiff_files = sorted(list(path.glob('*.tif')) + list(path.glob('*.tiff')))
        print(f"   - {path.name}: {len(tiff_files)} TIFF files")
    
    if len(valid_paths) > 3:
        print(f"   ... and {len(valid_paths) - 3} more folders")
    
    # Auto-detect parameters from first folder
    if valid_paths:
        tomo_params = load_tomography_params(valid_paths[0])
        NUM_DARKS = tomo_params.get('num_darks', 10)
        NUM_FLATS = tomo_params.get('num_flats', 10)
        NUM_PROJECTIONS = tomo_params.get('num_projections', 10)
        
        print(f"\n📊 Using parameters from {valid_paths[0].name}:")
        print(f"   Darks: {NUM_DARKS}")
        print(f"   Flats: {NUM_FLATS}")
        print(f"   Projections per folder: {NUM_PROJECTIONS}")
    else:
        print("❌ No valid folders found")
        NUM_DARKS = 10
        NUM_FLATS = 10
        NUM_PROJECTIONS = 0
else:
    # Single folder
    if DATA_PATH.exists():
        tiff_files = sorted(list(DATA_PATH.glob('*.tif')) + list(DATA_PATH.glob('*.tiff')))
        print(f"✅ Found {len(tiff_files)} TIFF files in {DATA_PATH.name}")
        
        # Auto-detect num_darks and num_flats from log file
        tomo_params = load_tomography_params(DATA_PATH)
        NUM_DARKS = tomo_params.get('num_darks', 10)
        NUM_FLATS = tomo_params.get('num_flats', 10)
        NUM_PROJECTIONS = tomo_params.get('num_projections', 10)
        
        print(f"\n📊 Detected tomography parameters:")
        print(f"   Darks: {NUM_DARKS}")
        print(f"   Flats: {NUM_FLATS}")
        print(f"   Projections: {NUM_PROJECTIONS}")
    else:
        print(f"❌ Path not found: {DATA_PATH}")
        print("   Please update DATA_PATH to point to your data folder.")
        NUM_DARKS = 10  # Fallback defaults
        NUM_FLATS = 10
        NUM_PROJECTIONS = 0

## 3. Understanding the TomographyFolderProcessor

The `TomographyFolderProcessor` handles loading and preprocessing of tomographic data from a single folder.

In [ ]:
# Create a folder processor for the first folder (for demonstration)
# Note: num_darks and num_flats can be set to None for auto-detection from log file

# Get the first valid folder
if isinstance(DATA_PATH, list):
    demo_folder = valid_paths[0] if valid_paths else None
else:
    demo_folder = DATA_PATH if DATA_PATH.exists() else None

if demo_folder is None:
    print("❌ No valid folders available")
else:
    processor = TomographyFolderProcessor(
        folder_path=demo_folder,
        num_darks=None,  # Auto-detect from log file
        num_flats=None,  # Auto-detect from log file
        epsilon=1e-6,
        cache_in_memory=False,  # Set to True for faster repeated access
        verbose=True,
        use_attenuation=False
    )
    
    print(f"\n📊 Folder Statistics:")
    print(f"   Original image size: {processor.original_width} × {processor.original_height}")
    print(f"   Number of projections: {processor.num_projections}")
    print(f"   Data type: {processor.dtype}")

In [ ]:
# Compute the range from the projections
if demo_folder is not None:
    global_min, global_max = processor.compute_range(
        sample_ratio=0.2,  # Sample 20% of projections
        max_samples=None
    )
    
    print(f"\n📈 Projection Range:")
    print(f"   Min: {global_min:.4f}")
    print(f"   Max: {global_max:.4f}")
    print(f"   Range: {global_max - global_min:.4f}")

In [ ]:
# Load and visualize a single projection
if demo_folder is not None:
    proj_idx = min(661, processor.num_projections - 1)  # Choose a projection index
    
    # Get the projection (normalized to [0, 1])
    projection = processor.get_projection(proj_idx, normalize=False)
    
    print(f"Projection {proj_idx}:")
    print(f"  Shape: {projection.shape}")
    print(f"  Range: [{projection.min():.4f}, {projection.max():.4f}]")
    
    # Visualize
    plt.figure(figsize=(8, 8))
    plt.imshow(projection)
    plt.colorbar(label='Normalized Intensity')
    plt.title(f'Projection {proj_idx} from {demo_folder.name}')
    plt.xlabel('X pixel')
    plt.ylabel('Y pixel')
    plt.show()

## 4. Creating the Dataset

The `ProjectionTripletDataset` creates sequences of consecutive projections for Noise2Noise training:
- **Input**: Stack of k consecutive projections P_i, P_{i+1}, ..., P_{i+k-1} (k channels)
- **Target**: Next projection P_{i+k} (1 channel)

The number of input projections k is controlled by the `num_input_projections` parameter (default=2).

In [ ]:
# Reload the dataset module to pick up any changes
import importlib
from sdate.datasets import projection_triplet_dataset
importlib.reload(projection_triplet_dataset)

# Re-import the classes
from sdate.datasets.projection_triplet_dataset import (
    ProjectionTripletDataset,
    TomographyFolderProcessor,
    create_train_val_split,
    load_tomography_params
)

print("✅ ProjectionTripletDataset module reloaded!")

In [ ]:
# Create the dataset
# Note: num_darks and num_flats can be set to None for auto-detection from log file

NUM_INPUT_PROJECTIONS = 3  # Number of input channels (k). Target is the (k+1)-th projection.

dataset = ProjectionTripletDataset(
    folder_paths=DATA_PATH,
    target_size=TARGET_SIZE,
    num_darks=None,  # Auto-detect from log file
    num_flats=None,  # Auto-detect from log file
    epsilon=1e-6,
    cache_in_memory=False,
    preload=False,
    augment=False,  # Set to True for training
    verbose=True,
    use_attenuation=False,
    num_input_projections=NUM_INPUT_PROJECTIONS
)

print(f"\n📊 Dataset Statistics:")
print(f"   Total samples: {len(dataset)}")
print(f"   Input projections per sample (k): {NUM_INPUT_PROJECTIONS}")
print(f"   Target size: {TARGET_SIZE} × {TARGET_SIZE}")

In [ ]:
sample_idx

In [ ]:
import random
# Get a sample from the dataset
sample_idx = random.randint(0, len(dataset) - 1)
sample = dataset[sample_idx]
print("sample_idx", sample_idx)

# Visualize the input projections and target
k = NUM_INPUT_PROJECTIONS
num_cols = k + 2  # k input channels + target + difference
fig, axes = plt.subplots(1, num_cols, figsize=(5 * num_cols, 5))

range_min = sample['input'].min()
range_max = sample['input'].max()

target_proj_idx = sample['projection_idx']

# Plot each input channel
for ch in range(k):
    axes[ch].imshow(sample['input'][ch].numpy(), vmin=range_min, vmax=range_max)
    input_proj_idx = target_proj_idx - k + ch
    axes[ch].set_title(f'P_{{{input_proj_idx}}} (Input Ch. {ch})')
    axes[ch].axis('off')

# Target projection
im_target = axes[k].imshow(sample['target'][0].numpy(), vmin=range_min, vmax=range_max)
axes[k].set_title(f'P_{{{target_proj_idx}}} (Target)')
axes[k].axis('off')
plt.colorbar(im_target, ax=axes[k], fraction=0.046)

# Difference between average of inputs and target
input_avg = sample['input'][0]
diff = input_avg - sample['target'][0]
vmax_diff = np.abs(diff).max()
im_diff = axes[k + 1].imshow(diff, cmap='RdBu', vmin=-vmax_diff, vmax=vmax_diff)
axes[k + 1].set_title(f'Avg(inputs) - Target\n(noise/motion)')
axes[k + 1].axis('off')
plt.colorbar(im_diff, ax=axes[k + 1], fraction=0.046)

plt.suptitle(f'Noise2Noise Visualization (k={k} input projections)\nCenter coords: ({sample["center_coords"][0]:.3f}, {sample["center_coords"][1]:.3f})', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
try:
    from ipywidgets import interact, FloatSlider
except ImportError:
    !pip install ipywidgets
    from ipywidgets import interact, FloatSlider
import torch.nn.functional as F

@interact(offset_x=FloatSlider(min=-5.0, max=5.0, step=0.1, value=0.0, description='Offset X (px)'),
          offset_y=FloatSlider(min=-5.0, max=5.0, step=0.1, value=0.0, description='Offset Y (px)'))
def visualize_offset(offset_x, offset_y):
    """
    Visualize how sub-pixel shifting of the last input frame affects the difference with target.
    
    Args:
        offset_x: Horizontal offset in pixels (-5 to +5)
        offset_y: Vertical offset in pixels (-5 to +5)
    """
    # Get the last input frame and target
    last_input = sample['input'][-1]  # Shape: (H, W)
    target = sample['target'][0]
    
    # Convert to torch tensors for interpolation
    last_input_t = last_input.unsqueeze(0).unsqueeze(0)
    target_t = target.unsqueeze(0).unsqueeze(0)
    
    # Create affine transformation matrix for translation
    # Normalize offset to [-1, 1] range (required by affine_grid)
    H, W = last_input.shape
    tx = 2 * offset_x / W  # Normalize to [-1, 1]
    ty = 2 * offset_y / H
    
    # Affine matrix: [[1, 0, tx], [0, 1, ty]]
    theta = torch.tensor([[[1.0, 0.0, tx],
                           [0.0, 1.0, ty]]], dtype=torch.float32)
    
    # Apply transformation
    grid = F.affine_grid(theta, last_input_t.size(), align_corners=False)
    shifted = F.grid_sample(last_input_t, grid, mode='bilinear', padding_mode='border', align_corners=False)
    
    # Compute difference
    diff = shifted - target_t
    
    # Convert back to numpy for visualization
    shifted_np = shifted[0, 0].cpu().numpy()
    diff_np = diff[0, 0].cpu().numpy()
    target_np = target.numpy()
    
    # Compute overall metrics
    mse = np.mean(diff_np ** 2)
    psnr = 10 * np.log10(1.0 / max(mse, 1e-10))
    
    # Compute 95th percentile threshold on absolute difference
    abs_diff = np.abs(diff_np)
    threshold_95 = np.percentile(abs_diff, 95)
    
    # Create masks for below and above 95th percentile
    mask_below_95 = abs_diff <= threshold_95
    mask_above_95 = abs_diff > threshold_95
    
    # Compute MSE and PSNR for pixels below 95th percentile
    if mask_below_95.sum() > 0:
        mse_below_95 = np.mean(diff_np[mask_below_95] ** 2)
        psnr_below_95 = 10 * np.log10(1.0 / max(mse_below_95, 1e-10))
    else:
        mse_below_95 = 0.0
        psnr_below_95 = float('inf')
    
    # Compute MSE and PSNR for pixels above 95th percentile
    if mask_above_95.sum() > 0:
        mse_above_95 = np.mean(diff_np[mask_above_95] ** 2)
        psnr_above_95 = 10 * np.log10(1.0 / max(mse_above_95, 1e-10))
    else:
        mse_above_95 = 0.0
        psnr_above_95 = float('inf')
    
    # Visualize
    fig, axes = plt.subplots(1, 4, figsize=(22, 5))
    
    # Shifted input
    axes[0].imshow(shifted_np, vmin=0, vmax=1)
    axes[0].set_title(f'Shifted Input\n(Δx={offset_x:.1f}px, Δy={offset_y:.1f}px)')
    axes[0].axis('off')
    
    # Target
    axes[1].imshow(target_np, vmin=0, vmax=1)
    axes[1].set_title('Target')
    axes[1].axis('off')
    
    # Difference
    vmax = max(np.abs(diff_np).max(), 1e-6)
    vmax = 0.03
    im = axes[2].imshow(diff_np, cmap='RdBu', vmin=-vmax, vmax=vmax)
    axes[2].set_title(f'Difference (Shifted - Target)\nMSE: {mse:.6f}, PSNR: {psnr:.2f} dB')
    axes[2].axis('off')
    plt.colorbar(im, ax=axes[2], fraction=0.046)
    
    # Mask visualization: show pixels above 95th percentile
    mask_viz = np.zeros((*diff_np.shape, 3))
    mask_viz[..., 0] = mask_above_95.astype(float)  # Red channel for above 95th
    mask_viz[..., 2] = mask_below_95.astype(float) * 0.3  # Blue channel (dimmed) for below 95th
    axes[3].imshow(mask_viz)
    axes[3].set_title(f'95th Percentile Mask\nRed: top 5% ({mask_above_95.sum()} px)')
    axes[3].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    # Print statistics
    print(f"Offset: ({offset_x:+.1f}px, {offset_y:+.1f}px)")
    print(f"\n{'='*60}")
    print(f"{'Metric':<25} {'Overall':>12} {'Below 95%':>12} {'Above 95%':>12}")
    print(f"{'-'*60}")
    print(f"{'MSE':<25} {mse:>12.8f} {mse_below_95:>12.8f} {mse_above_95:>12.8f}")
    print(f"{'PSNR (dB)':<25} {psnr:>12.2f} {psnr_below_95:>12.2f} {psnr_above_95:>12.2f}")
    print(f"{'Pixel Count':<25} {diff_np.size:>12} {mask_below_95.sum():>12} {mask_above_95.sum():>12}")
    print(f"{'='*60}")
    print(f"\n95th percentile threshold: {threshold_95:.6f}")
    print(f"Max absolute error: {np.abs(diff_np).max():.6f}")

## 8. Model Setup (UNet2DModel from Diffusers)

In [ ]:
from diffusers import UNet2DModel

# Create the UNet model
model = UNet2DModel(
    sample_size=TARGET_SIZE,  # the target image resolution
    in_channels=NUM_INPUT_PROJECTIONS,  # k input channels: P_i, P_{i+1}, ..., P_{i+k-1}
    out_channels=1,  # 1 output channel: predicted P_{i+k}
    layers_per_block=2,  # ResNet layers per UNet block
    block_out_channels=(64, 64, 128, 128, 256, 256),  # channels for each block
    down_block_types=(
        "DownBlock2D",
        "DownBlock2D",
        "DownBlock2D",
        "DownBlock2D",
        "AttnDownBlock2D",  # attention block
        "DownBlock2D",
    ),
    up_block_types=(
        "UpBlock2D",
        "AttnUpBlock2D",  # attention block
        "UpBlock2D",
        "UpBlock2D",
        "UpBlock2D",
        "UpBlock2D",
    ),
)

# Count parameters
num_params = sum(p.numel() for p in model.parameters())
num_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"🏗️  Model Summary:")
print(f"   Total parameters: {num_params:,}")
print(f"   Trainable parameters: {num_trainable:,}")
print(f"   Input channels: {NUM_INPUT_PROJECTIONS}")
print(f"   Output channels: 1")
print(f"   Sample size: {TARGET_SIZE}×{TARGET_SIZE}")

# Load trained model weights if available
checkpoint_path = Path("../checkpoints/best_model.pt")
if checkpoint_path.exists():
    print(f"\n📂 Loading trained model from {checkpoint_path}")
    checkpoint = torch.load(checkpoint_path, map_location='cpu')
    model.load_state_dict(checkpoint['model_state_dict'])
    epoch = checkpoint['epoch']
    val_loss = checkpoint.get('best_val_loss', 'N/A')
    print(f"   ✅ Loaded checkpoint from epoch {epoch}")
    print(f"   Validation loss: {val_loss}")
else:
    print(f"\n⚠️  No trained model found at {checkpoint_path}")
    print(f"   Using untrained model (random weights)")

# Test forward pass
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

model = model.to(device)

## 10. Visualize Model Prediction (Untrained)

In [ ]:
# Get a prediction from the (mostly untrained) model
model.eval()
DRIFT_MODES = ["phase_correlation", "optical_flow", "cross_correlation"]
with torch.no_grad():
    sample = dataset[3028]
    input_tensor = sample['input'].unsqueeze(0).to(device)
    target_tensor = sample['target'].unsqueeze(0).to(device)
    center_coords = sample['center_coords'].unsqueeze(0).to(device)  # (B, 2)
    conditioning = center_coords[:, 0].long()  # Use the first coordinate as timestep to condition on the x position
    
    # conditioning = torch.zeros(1, dtype=torch.long, device=device)
    
    pred = model(input_tensor, conditioning, return_dict=False)[0]
    
    # Compute MSE
    mse = torch.nn.functional.mse_loss(pred, target_tensor).item()

# Get drift predictor prediction (phase correlation)
from compress_ct.drift_predictor import DriftPredictor
drift_pred = DriftPredictor(mode=DRIFT_MODES[2], num_input_projections=NUM_INPUT_PROJECTIONS, verbose=False)
prev_frames = [sample['input'][i].numpy() for i in range(NUM_INPUT_PROJECTIONS)]
drift_prediction = drift_pred.predict_frame(prev_frames)

# Visualize
k = NUM_INPUT_PROJECTIONS
num_top_cols = k + 1  # k inputs + average
num_bot_cols = 3      # target, prediction, difference
num_cols = max(num_top_cols, num_bot_cols)
fig, axes = plt.subplots(2, num_cols, figsize=(5 * num_cols, 10))

# Hide unused axes
for ax_row in axes:
    for ax in ax_row:
        ax.axis('off')

# Top row: input channels + average
for ch in range(k):
    axes[0, ch].imshow(sample['input'][ch].numpy(), vmin=0, vmax=1)
    axes[0, ch].set_title(f'Input Ch. {ch}: P_{{i+{ch}}}')
    axes[0, ch].axis('off')
padding = 6
# Drift difference
diff_drift = (drift_prediction[:, padding:-padding] - sample['target'][0][:, padding:-padding].numpy())
vmax_drift = np.abs(diff_drift).max()
im = axes[0, k].imshow(diff_drift, cmap='RdBu', vmin=-vmax_drift, vmax=vmax_drift)
axes[0, k].set_title('Drift Diff\n(Pred - Target)')
axes[0, k].axis('off')
plt.colorbar(im, ax=axes[0, k], fraction=0.046)

# Bottom row: target, prediction, difference
axes[1, 0].imshow(sample['target'][0].numpy(), vmin=0, vmax=1)
axes[1, 0].set_title('Target: P_{i+k} (Ground Truth)')
axes[1, 0].axis('off')

im = axes [1, 1]. imshow(pred [0, 0] .cpu().numpy(), vmin=0, vmax=1)
axes [1, 1]. imshow(pred [0, 0].cpu( ).numpy(), vmin=0, vmax=1)
axes [1, 1].set_title(f'Prediction (Untrained) \nMSE: {mse: .6f}')
axes [1, 1].axis( 'off')
# Difference

diff = (pred[0, 0, :, padding:-padding].cpu() - sample['target'][0][:, padding:-padding]).numpy()
vmax = np.abs(diff).max()
im = axes[1, 2].imshow(diff, cmap='RdBu', vmin=-vmax, vmax=vmax)
axes[1, 2].set_title('Difference (Pred - Target)')
axes[1, 2].axis('off')
plt.colorbar(im, ax=axes[1, 2], fraction=0.046)

diff2 = (sample['input'][-1][:, padding:-padding] - sample['target'][0][:, padding:-padding]).numpy()
vmax = np.abs(diff2).max()
im = axes[1, 3].imshow(diff2, cmap='RdBu', vmin=-vmax, vmax=vmax)
axes[1, 3].set_title('Difference (Input - Target)')
axes[1, 3].axis('off')
plt.colorbar(im, ax=axes[1, 3], fraction=0.046)

plt.suptitle(f'Noise2Noise Model Prediction (Untrained, k={k})', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 12. Summary and Next Steps

In [ ]:
import sys, tempfile 
from pathlib import Path
sys.path. insert(0, str(Path.cwd().parent) )
from compress_ct import CTCompressor, CTDecompressor 
from compress_ct.predictor import BlockPredictor
# ---- Build the predictor from the trained model ----
# Re-use the model object already loaded above (Section 8).
# It was trained at target_size=512, but we feed 256x256 patches.
predictor = BlockPredictor(
    model=model, 
    block_size=256, 
    num_input_projections=NUM_INPUT_PROJECTIONS, 
    device=device, overlap=32,
)
# --- Collect a small sequence of frames from the dataset ----
# We'll take 8 consecutive projections from the first folder
n_demo_frames = 8
processor = dataset.processors[1]
demo_frames = [
    processor.get_projection(20 + i, normalize=True) 
    for i in range(n_demo_frames)
]
print(f"Collected {len(demo_frames)} frames, shape {demo_frames[0].shape}")
# ---- Compress ----
compressor = CTCompressor (
    predictor=predictor, 
    patch_size=256, 
    residual_quality=100, 
    jpeg_quality=100, 
    dct_block_size=8,
    fallback_threshold=None, # keep whichever is smaller automatically 
    verbose=True,
)
archive_path = Path(tempfile.mktemp(suffix=".ctc"))
stats = compressor.compress(demo_frames, archive_path)
print(f"\nArchive size: {archive_path.stat().st_size:,} bytes")
print(f"Compression ratio: {stats ['compression_ratio']:.2f}x")

In [ ]:
# -- Decompress ----
decompressor = CTDecompressor (predictor=predictor, verbose=True)
recovered = decompressor.decompress(archive_path)
# ---- Compare original vs. reconstructed ----
n_show = min(4, n_demo_frames)
fig, axes = plt.subplots(3, n_show, figsize=(5 * n_show, 14))
for i, j in enumerate(range(4, 4+n_show)):
    # Original
    axes[0, i].imshow(demo_frames[j], vmin=0, vmax=1)
    axes[0, i].set_title(f"Original #{j}")
    axes[0, i].axis("off")
    # Reconstructed
    axes[1, i].imshow(recovered[j], vmin=0, vmax=1)
    mse_val = np.mean((demo_frames[j] - recovered[j]) ** 2)
    psnr_val = 10 * np.log10(1.0 / max(mse_val, 1e-10))
    axes[1, i].set_title(f"Reconstructed #{j}\nPSNR {psnr_val:.1f} dB")
    axes[1, i].axis("off")
    # Difference (amplified)
    diff = demo_frames[j] - recovered[j]
    vmax = max(np.abs(diff).max(), 1e-6)
    im = axes[2, i].imshow(diff, cmap="RdBu", vmin=-vmax, vmax=vmax)
    axes[2, i].set_title(f"Difference #{i}")
    axes[2, i].axis("off")
    plt.colorbar(im, ax=axes[2, i], fraction=0.046)
plt.suptitle("Neural CT Compression: Original → Compressed → Reconstructed", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()
# Print per-frame stats
print ("\nPer-frame statistics:")
print (f"{'Frame':>6} {'Bytes':>10} {'Residual':>10} {'JPEG':>10} {'JPEG FB':>10}")
for i in range(len(stats ['frame_bytes'])):
    fb = stats ['fallback_counts'] [i]
    res = stats ['residual_counts'] [i]
    b = stats ['frame_bytes'] [i]
    print(f"{i:6d} {b:10d} {res:10d} {fb:10d}")
# Cleanup
# archive_path.unlink(missing_ok=True)

## 13. Compare with HVEC Compression

Now let's compare our custom neural compressor with HVEC (H.265) compression using the same demo frames.

In [ ]:
from sdate.stream_hvec.stream_gray10 import HevcGray10Streamer, EncoderParams
import tempfile
import os

n_demo_frames = 20

demo_frames = [
    processor.get_projection(20 + i, normalize=True) 
    for i in range(n_demo_frames)
]

# Create a temporary directory for HVEC output
hvec_temp_dir = Path(tempfile.mkdtemp())
hvec_output = hvec_temp_dir / "hvec_compressed.mov"

# Convert frames to torch tensors if needed
demo_frames_torch = torch.stack([torch.tensor(f, dtype=torch.float32) for f in demo_frames])

# demo_frames_torch = demo_frames_torch_all[1:] - demo_frames_torch_all[:-1]
# demo_frames_torch -= demo_frames_torch.min()
# demo_frames_torch /= demo_frames_torch.max()

print(f"🎥 HVEC Compression with quality=90...")
print(f"   Input: {len(demo_frames_torch)} frames, shape {demo_frames_torch[0].shape}")

# Create HVEC streamer with quality 90 (similar to our custom compressor)
# Force software encoding since hardware encoder is not available on Linux
quality = 95
params = EncoderParams(
    fps=24,
    cq_hw=quality,  # Hardware quality (not used in software mode)
    crf_sw=min(51, 51 - int(min(quality, 100) * 51 / 100)),  # Software quality - lower is better (0-51)
    preset_sw="slow",
    force_software=True  # Force software encoding (libx265)
)

streamer = HevcGray10Streamer(
    base_path=hvec_temp_dir,
    segment_prefix="demo",
    params=params
)

# Compress all frames to a single segment
with streamer.start_segment(q=90):
    for i, frame in enumerate(demo_frames_torch):
        streamer.append_frame(frame)
        if (i + 1) % 2 == 0:
            print(f"   Encoded frame {i+1}/{len(demo_frames_torch)}")

# Get the output file (first segment)
hvec_file = streamer.segments[0]
hvec_size = hvec_file.stat().st_size

print(f"\n✅ HVEC Compression Complete!")
print(f"   Output file: {hvec_file.name}")
print(f"   File size: {hvec_size:,} bytes")
print(f"   Using: {'Hardware' if streamer._using_hardware else 'Software'} encoder")

In [ ]:
# Calculate compression ratios for comparison
original_size_per_frame = demo_frames[0].nbytes
total_original_size = original_size_per_frame * len(demo_frames)

# Custom compressor stats (from earlier)
custom_size = archive_path.stat().st_size
custom_ratio = stats['compression_ratio']

# HVEC stats
hvec_ratio = total_original_size / hvec_size

print("=" * 80)
print("📊 COMPRESSION COMPARISON")
print("=" * 80)
print(f"\n🔢 Original Data:")
print(f"   Frames: {len(demo_frames)}")
print(f"   Size per frame: {original_size_per_frame:,} bytes ({demo_frames[0].shape})")
print(f"   Total size: {total_original_size:,} bytes ({total_original_size / 1e6:.2f} MB)")
print(f"\n🧠 Custom Neural Compressor (with Predictor):")
print(f"   Compressed size: {custom_size:,} bytes ({custom_size / 1e6:.2f} MB)")
print(f"   Compression ratio: {custom_ratio:.2f}x")
print(f"   Space savings: {100 * (1 - custom_size/total_original_size):.1f}%")
print(f"\n🎥 HVEC (H.265) Compression:")
print(f"   Compressed size: {hvec_size:,} bytes ({hvec_size / 1e6:.2f} MB)")
print(f"   Compression ratio: {hvec_ratio:.2f}x")
print(f"   Space savings: {100 * (1 - hvec_size/total_original_size):.1f}%")
print(f"\n⚖️  Comparison:")
if custom_size < hvec_size:
    factor = hvec_size / custom_size
    print(f"   🏆 Custom compressor is {factor:.2f}x SMALLER than HVEC")
    print(f"   ({(hvec_size - custom_size):,} bytes smaller)")
else:
    factor = custom_size / hvec_size
    print(f"   🏆 HVEC is {factor:.2f}x SMALLER than custom compressor")
    print(f"   ({(custom_size - hvec_size):,} bytes smaller)")
print("=" * 80)

In [ ]:
# Decompress HVEC for quality comparison
import cv2

# Read the HVEC compressed video
cap = cv2.VideoCapture(str(hvec_file))

hvec_recovered = []
while True:
    ret, frame = cap.read()
    if not ret:
        break
    # Convert from BGR to grayscale (HVEC stores as grayscale in Y channel)
    # The frame is 10-bit but cv2 returns 8-bit, we need to normalize
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    # Normalize to [0, 1] range
    hvec_recovered.append(gray.astype(np.float32) / 255.0)

cap.release()

print(f"✅ Decompressed {len(hvec_recovered)} frames from HVEC")
print(f"   Frame shape: {hvec_recovered[0].shape}")

# Compute PSNR for both methods
def compute_psnr(original, reconstructed):
    mse = np.mean((original - reconstructed) ** 2)
    if mse == 0:
        return float('inf')
    return 10 * np.log10(1.0 / mse)

print(f"\n📈 Quality Comparison (PSNR):")
print(f"{'Frame':>6} {'Custom PSNR':>12} {'HVEC PSNR':>12}")
print("-" * 35)

custom_psnrs = []
hvec_psnrs = []

for i in range(4, min(len(demo_frames), len(recovered), len(hvec_recovered))):
    custom_psnr = compute_psnr(demo_frames[i], recovered[i])
    hvec_psnr = compute_psnr(demo_frames[i], hvec_recovered[i])
    custom_psnrs.append(custom_psnr)
    hvec_psnrs.append(hvec_psnr)
    print(f"{i:6d} {custom_psnr:12.2f} {hvec_psnr:12.2f}")

print("-" * 35)
print(f"{'Avg':>6} {np.mean(custom_psnrs):12.2f} {np.mean(hvec_psnrs):12.2f}")

In [ ]:
# Visualize side-by-side comparison
fig, axes = plt.subplots(3, 4, figsize=(20, 14))

frame_indices = list(range(4, 8))  # Show frames 4-7

for col, i in enumerate(frame_indices):
    # Original
    axes[0, col].imshow(demo_frames[i], vmin=0, vmax=1)
    axes[0, col].set_title(f"Original #{i}")
    axes[0, col].axis("off")
    
    # Custom compressor
    custom_psnr = compute_psnr(demo_frames[i], recovered[i])
    axes[1, col].imshow(recovered[i], vmin=0, vmax=1)
    axes[1, col].set_title(f"Custom (Neural)\nPSNR: {custom_psnr:.1f} dB")
    axes[1, col].axis("off")
    
    # HVEC
    hvec_psnr = compute_psnr(demo_frames[i], hvec_recovered[i])
    axes[2, col].imshow(hvec_recovered[i], vmin=0, vmax=1)
    axes[2, col].set_title(f"HVEC (H.265)\nPSNR: {hvec_psnr:.1f} dB")
    axes[2, col].axis("off")

plt.suptitle(
    f"Compression Comparison: Custom Neural ({custom_ratio:.1f}x, {custom_size/1e6:.1f}MB) "
    f"vs HVEC ({hvec_ratio:.1f}x, {hvec_size/1e6:.1f}MB)",
    fontsize=14, 
    fontweight="bold"
)
plt.tight_layout()
plt.show()

## 14. Compression Analysis Summary

### Key Findings:

#### Compression Ratio:
- **HVEC (H.265)**: 106.5x compression (1.66 MB)
- **Custom Neural**: 16.75x compression (10.56 MB)
- **Winner**: HVEC is **6.36x smaller** than the custom compressor

#### Quality (PSNR):
- **HVEC (H.265)**: 44.81 dB average
- **Custom Neural**: 44.46 dB average
- **Winner**: Comparable quality, HVEC slightly better (+0.35 dB)

### Why is HVEC More Efficient?

1. **Mature Industry Standard**: H.265/HEVC is highly optimized after years of development
2. **Temporal Redundancy**: HVEC excels at exploiting temporal correlations between frames
3. **Sophisticated Prediction**: Uses motion compensation and inter-frame prediction
4. **Optimized Entropy Coding**: Context-adaptive binary arithmetic coding (CABAC)

### Custom Neural Compressor Advantages:

1. **Task-Specific Learning**: Can be trained on specific CT/tomography data characteristics
2. **Lossless Residuals**: Option for lossless compression of prediction residuals
3. **Flexible Quality Control**: Fine-grained control over compression parameters
4. **No Temporal Limitations**: Works on any frame order, no GOP restrictions
5. **Research Platform**: Easier to experiment with novel prediction architectures

### Recommendations:

- **For Production**: Use HVEC for best compression ratio with excellent quality
- **For Research**: Use custom neural compressor to explore domain-specific optimizations
- **Hybrid Approach**: Use neural predictor with HVEC encoding of residuals

In [ ]:
# Create a detailed comparison plot showing differences
fig, axes = plt.subplots(2, 4, figsize=(20, 10))

frame_indices = list(range(4, 8))

for col, i in enumerate(frame_indices):
    # Custom compressor difference
    custom_diff = demo_frames[i] - recovered[i]
    vmax_custom = max(np.abs(custom_diff).max(), 1e-6)
    im1 = axes[0, col].imshow(custom_diff, cmap='RdBu', vmin=-vmax_custom, vmax=vmax_custom)
    custom_psnr = compute_psnr(demo_frames[i], recovered[i])
    axes[0, col].set_title(f"Custom Error #{i}\nPSNR: {custom_psnr:.2f} dB")
    axes[0, col].axis("off")
    plt.colorbar(im1, ax=axes[0, col], fraction=0.046)
    
    # HVEC difference
    hvec_diff = demo_frames[i] - hvec_recovered[i]
    vmax_hvec = max(np.abs(hvec_diff).max(), 1e-6)
    im2 = axes[1, col].imshow(hvec_diff, cmap='RdBu', vmin=-vmax_hvec, vmax=vmax_hvec)
    hvec_psnr = compute_psnr(demo_frames[i], hvec_recovered[i])
    axes[1, col].set_title(f"HVEC Error #{i}\nPSNR: {hvec_psnr:.2f} dB")
    axes[1, col].axis("off")
    plt.colorbar(im2, ax=axes[1, col], fraction=0.046)

plt.suptitle("Reconstruction Error Comparison (Difference from Original)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

# Print statistics
print("\n" + "="*60)
print("ERROR STATISTICS")
print("="*60)
print(f"{'Frame':>6} {'Custom MAE':>12} {'HVEC MAE':>12} {'Custom Max':>12} {'HVEC Max':>12}")
print("-"*60)
for i in range(4, 8):
    custom_diff = np.abs(demo_frames[i] - recovered[i])
    hvec_diff = np.abs(demo_frames[i] - hvec_recovered[i])
    print(f"{i:6d} {custom_diff.mean():12.6f} {hvec_diff.mean():12.6f} {custom_diff.max():12.6f} {hvec_diff.max():12.6f}")
print("="*60)

In [ ]:
plt.figure(figsize=(10, 10))
plt.imshow(demo_frames[j])
plt.colorbar()

In [ ]:
plt.figure(figsize=(10, 10))
plt.imshow(torch.tensor(recovered[j]))
plt.colorbar()

## 15. Drift-Based Predictor — Setup & Visual Drift Check

The `DriftPredictor` estimates the translation (drift) between two consecutive
frames and extrapolates it to predict the next frame.  No neural network needed.

Three modes:
- **`phase_correlation`** — sub-pixel FFT-based (recommended, fast)
- **`optical_flow`** — dense Farnebäck flow (slower, richer)
- **`cross_correlation`** — template matching (integer-pixel)

In [ ]:
import importlib, compress_ct.drift_predictor, compress_ct
importlib.reload(compress_ct.drift_predictor)
importlib.reload(compress_ct)

from compress_ct.drift_predictor import DriftPredictor, DriftMode
from compress_ct import CTCompressor, CTDecompressor

# ----- Quick sanity check: visualise the drift between two frames -----
idx_start = 20
x_prev = processor.get_projection(idx_start, normalize=True)
x_last = processor.get_projection(idx_start + 1, normalize=True)
x_target = processor.get_projection(idx_start + 2, normalize=True)

# Estimate drift with phase correlation
dp = DriftPredictor(mode="phase_correlation", num_input_projections=NUM_INPUT_PROJECTIONS, verbose=True)
x_pred = dp.predict_frame([x_prev, x_last])

# Compare
mse_nopred = np.mean((x_last - x_target) ** 2)
mse_drift  = np.mean((x_pred - x_target) ** 2)
psnr_nopred = 10 * np.log10(1.0 / max(mse_nopred, 1e-10))
psnr_drift  = 10 * np.log10(1.0 / max(mse_drift, 1e-10))

print(f"\n📈 Drift prediction vs. no prediction:")
print(f"   No prediction  MSE={mse_nopred:.6f}  PSNR={psnr_nopred:.2f} dB")
print(f"   Drift predict  MSE={mse_drift:.6f}   PSNR={psnr_drift:.2f} dB")

fig, axes = plt.subplots(1, 4, figsize=(22, 5))
axes[0].imshow(x_last, vmin=0, vmax=1); axes[0].set_title("x_{-1} (last input)"); axes[0].axis("off")
axes[1].imshow(x_pred, vmin=0, vmax=1); axes[1].set_title(f"Drift prediction\nPSNR {psnr_drift:.1f} dB"); axes[1].axis("off")
axes[2].imshow(x_target, vmin=0, vmax=1); axes[2].set_title("x_0 (target)"); axes[2].axis("off")

diff = x_pred - x_target
vmax = max(np.abs(diff).max(), 1e-6)
im = axes[3].imshow(diff, cmap="RdBu", vmin=-vmax, vmax=vmax)
axes[3].set_title("Pred − Target"); axes[3].axis("off")
plt.colorbar(im, ax=axes[3], fraction=0.046)
plt.suptitle("Drift-Based Prediction Sanity Check (phase_correlation)", fontsize=14, fontweight="bold")
plt.tight_layout(); plt.show()

## 16. Compare All Drift Modes — Compression Ratio & Quality

Run the **same** compression pipeline on the same 8-frame demo sequence
using each drift mode, then compare against the neural predictor and HVEC.

In [ ]:
import importlib, compress_ct.drift_predictor, compress_ct.compressor, compress_ct.decompressor, compress_ct
importlib.reload(compress_ct.drift_predictor)
importlib.reload(compress_ct.compressor)
importlib.reload(compress_ct.decompressor)
importlib.reload(compress_ct)

from compress_ct.drift_predictor import DriftPredictor, DriftMode
from compress_ct import CTCompressor, CTDecompressor

import tempfile, time

# Drift modes to test
DRIFT_MODES = ["phase_correlation", "optical_flow", "cross_correlation"]

# Store results for each mode
results = {}

for mode in DRIFT_MODES:
    print(f"\n{'='*60}")
    print(f"  Mode: {mode}")
    print(f"{'='*60}")
    
    # Build predictor
    drift_pred = DriftPredictor(
        mode=mode,
        num_input_projections=NUM_INPUT_PROJECTIONS,
        verbose=True,
    )
    
    # Build compressor
    comp = CTCompressor(
        predictor=drift_pred,
        patch_size=256,
        residual_quality=100,
        jpeg_quality=100,
        dct_block_size=8,
        fallback_threshold=None,
        verbose=True,
    )
    
    # Compress
    out_path = Path(tempfile.mktemp(suffix=".ctc"))
    t0 = time.time()
    st = comp.compress(demo_frames, out_path)
    t_compress = time.time() - t0
    
    # Decompress
    decomp = CTDecompressor(predictor=drift_pred, verbose=True)
    t0 = time.time()
    rec = decomp.decompress(out_path)
    t_decompress = time.time() - t0
    
    # Quality
    psnrs = []
    for j in range(NUM_INPUT_PROJECTIONS, len(demo_frames)):
        mse_val = np.mean((demo_frames[j] - rec[j]) ** 2)
        psnrs.append(10 * np.log10(1.0 / max(mse_val, 1e-10)))
    
    results[mode] = {
        "archive_size": out_path.stat().st_size,
        "compression_ratio": st["compression_ratio"],
        "avg_psnr": np.mean(psnrs),
        "psnrs": psnrs,
        "recovered": rec,
        "stats": st,
        "archive_path": out_path,
        "t_compress": t_compress,
        "t_decompress": t_decompress,
    }
    
    print(f"  Archive: {out_path.stat().st_size:,} bytes  |  "
          f"Ratio: {st['compression_ratio']:.2f}x  |  "
          f"Avg PSNR: {np.mean(psnrs):.2f} dB  |  "
          f"Compress: {t_compress:.2f}s  Decompress: {t_decompress:.2f}s")

print("\n✅ All drift modes tested!")

In [ ]:
# ---- Summary table: all methods side by side ----
total_original = demo_frames[0].nbytes * len(demo_frames)

rows = []
# Neural predictor (from section 12)
rows.append(("Neural (UNet)", archive_path.stat().st_size,
             stats["compression_ratio"],
             np.mean([compute_psnr(demo_frames[i], recovered[i]) for i in range(NUM_INPUT_PROJECTIONS, len(demo_frames))])))

# Drift modes
for mode in DRIFT_MODES:
    r = results[mode]
    rows.append((f"Drift ({mode})", r["archive_size"], r["compression_ratio"], r["avg_psnr"]))

# HVEC
rows.append(("HVEC (H.265)", hvec_size, total_original / hvec_size,
             np.mean([compute_psnr(demo_frames[i], hvec_recovered[i])
                      for i in range(NUM_INPUT_PROJECTIONS, min(len(demo_frames), len(hvec_recovered)))])))

print("=" * 90)
print(f"{'Method':<30} {'Size (bytes)':>14} {'Ratio':>8} {'Avg PSNR (dB)':>14}")
print("-" * 90)
for name, size, ratio, psnr in rows:
    print(f"{name:<30} {size:>14,} {ratio:>8.2f}x {psnr:>14.2f}")
print("=" * 90)
print(f"\nOriginal data: {total_original:,} bytes ({total_original/1e6:.2f} MB)")

In [ ]:
# ---- Visual comparison: original vs drift-predicted reconstructions ----
n_show = min(4, len(demo_frames) - NUM_INPUT_PROJECTIONS)
frame_indices = list(range(NUM_INPUT_PROJECTIONS, NUM_INPUT_PROJECTIONS + n_show))

n_methods = len(DRIFT_MODES) + 1  # +1 for original row
fig, axes = plt.subplots(n_methods, n_show, figsize=(5 * n_show, 4.5 * n_methods))

for col, fi in enumerate(frame_indices):
    # Row 0: original
    axes[0, col].imshow(demo_frames[fi], vmin=0, vmax=1)
    axes[0, col].set_title(f"Original #{fi}")
    axes[0, col].axis("off")
    
    # Rows 1+: each drift mode
    for row, mode in enumerate(DRIFT_MODES, start=1):
        rec = results[mode]["recovered"]
        psnr = compute_psnr(demo_frames[fi], rec[fi])
        axes[row, col].imshow(rec[fi], vmin=0, vmax=1)
        axes[row, col].set_title(f"{mode}\nPSNR {psnr:.1f} dB")
        axes[row, col].axis("off")

plt.suptitle("Drift-Based Compression: Original vs Reconstructed", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# ---- Error maps for each drift mode ----
fig, axes = plt.subplots(len(DRIFT_MODES), n_show, figsize=(5 * n_show, 4 * len(DRIFT_MODES)))

for row, mode in enumerate(DRIFT_MODES):
    rec = results[mode]["recovered"]
    for col, fi in enumerate(frame_indices):
        diff = demo_frames[fi] - rec[fi]
        vmax = max(np.abs(diff).max(), 1e-6)
        im = axes[row, col].imshow(diff, cmap="RdBu", vmin=-vmax, vmax=vmax)
        psnr = compute_psnr(demo_frames[fi], rec[fi])
        axes[row, col].set_title(f"{mode} #{fi}\nPSNR {psnr:.2f} dB")
        axes[row, col].axis("off")
        plt.colorbar(im, ax=axes[row, col], fraction=0.046)

plt.suptitle("Reconstruction Error Maps by Drift Mode", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

# ---- Per-frame PSNR line plot ----
fig, ax = plt.subplots(figsize=(10, 5))
for mode in DRIFT_MODES:
    r = results[mode]
    xs = list(range(NUM_INPUT_PROJECTIONS, NUM_INPUT_PROJECTIONS + len(r["psnrs"])))
    ax.plot(xs, r["psnrs"], "o-", label=f"Drift ({mode})")

# Neural
neural_psnrs = [compute_psnr(demo_frames[i], recovered[i]) for i in range(NUM_INPUT_PROJECTIONS, len(demo_frames))]
ax.plot(range(NUM_INPUT_PROJECTIONS, NUM_INPUT_PROJECTIONS + len(neural_psnrs)), neural_psnrs, "s--", label="Neural (UNet)")

ax.set_xlabel("Frame index")
ax.set_ylabel("PSNR (dB)")
ax.set_title("Per-Frame PSNR: Drift Modes vs Neural Predictor")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Cleanup temp files
for mode in DRIFT_MODES:
    results[mode]["archive_path"].unlink(missing_ok=True)

## 17. Learnable Affine Warping for Frame Prediction

Instead of using fixed drift estimation, we can **learn** the optimal affine transformation
that warps `x_prev` into `x_target` using PyTorch's `affine_grid` + `grid_sample`.

The affine transformation matrix has 6 learnable parameters:
$$
\begin{bmatrix} a & b & t_x \\ c & d & t_y \end{bmatrix}
$$

We'll use gradient descent to minimize the MSE between the warped frame and the target.

In [ ]:
dataset.processors[3]._load_darks_flats()

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# ---- Get a pair of consecutive frames ----
proc_idx = 2
processor = dataset.processors[proc_idx]
if not hasattr(dataset.processors[proc_idx], "_flat_first"):
    dataset.processors[proc_idx]._load_darks_flats()
idx_start = 400
x_prev = processor.get_projection(idx_start, normalize=True)
x_target = processor.get_projection(idx_start + 1, normalize=True)

print(f"Frame shapes: x_prev={x_prev.shape}, x_target={x_target.shape}")

# Convert to torch tensors (B, C, H, W) format
x_prev_t = torch.tensor(x_prev, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device)
x_target_t = torch.tensor(x_target, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device)

print(f"Tensor shapes: x_prev_t={x_prev_t.shape}, x_target_t={x_target_t.shape}")


class LearnableAffineWarp(torch.nn.Module):
    """
    Learns an affine transformation to warp x_prev into x_target.
    
    The affine matrix is parameterized as:
    [[1+a, b, tx],
     [c, 1+d, ty]]
    
    Starting from identity (a=b=c=d=tx=ty=0).
    """
    def __init__(self):
        super().__init__()
        # Initialize to identity transform (small perturbations from identity)
        # We parameterize as offsets from identity for better optimization
        self.a = torch.nn.Parameter(torch.zeros(1))   # scale_x - 1
        self.b = torch.nn.Parameter(torch.zeros(1))   # shear_x
        self.c = torch.nn.Parameter(torch.zeros(1))   # shear_y  
        self.d = torch.nn.Parameter(torch.zeros(1))   # scale_y - 1
        self.tx = torch.nn.Parameter(torch.zeros(1))  # translate_x
        self.ty = torch.nn.Parameter(torch.zeros(1))  # translate_y
    
    def get_theta(self):
        """Build the 2x3 affine matrix."""
        theta = torch.zeros(1, 2, 3, device=self.a.device)
        theta[0, 0, 0] = 1 + self.a  # scale_x
        theta[0, 0, 1] = self.b       # shear_x
        theta[0, 0, 2] = self.tx      # translate_x
        theta[0, 1, 0] = self.c       # shear_y
        theta[0, 1, 1] = 1 + self.d   # scale_y
        theta[0, 1, 2] = self.ty      # translate_y
        return theta
    
    def forward(self, x):
        """Warp input x using learned affine transform."""
        theta = self.get_theta()
        # Generate sampling grid
        grid = F.affine_grid(theta, x.size(), align_corners=False)
        # Sample from input using the grid
        warped = F.grid_sample(x, grid, mode='bilinear', padding_mode='border', align_corners=False)
        return warped


def masked_loss(pred, target, use_mask=True, percentile=95):
    """
    Compute L1 loss with optional masking of outliers above the given percentile.
    
    Args:
        pred: Predicted tensor
        target: Target tensor
        use_mask: Whether to apply the mask
        percentile: Percentile threshold (values above this are masked out)
    
    Returns:
        loss: Masked L1 loss
        mask: The mask used (for visualization)
    """
    diff = torch.abs(pred - target)
    
    if use_mask:
        # Compute the percentile threshold
        threshold = torch.quantile(diff.flatten(), percentile / 100.0)
        # Create mask: True for values to KEEP (below threshold)
        mask = diff <= threshold
        # Compute masked loss
        masked_diff = diff * mask.float()
        loss = masked_diff.sum() / mask.sum().clamp(min=1)
    else:
        mask = torch.ones_like(diff, dtype=torch.bool)
        loss = diff.mean()
    
    return loss, mask


# ---- Create the model and optimizer ----
warp_model = LearnableAffineWarp().to(device)
optimizer = torch.optim.Adam(warp_model.parameters(), lr=0.0001)

# ---- Training loop ----
n_iterations = 300
warmup_iterations = 50  # No mask during warmup
losses = []
param_history = {k: [] for k in ['a', 'b', 'c', 'd', 'tx', 'ty']}
mask_history = []  # Track mask usage

print("\n🔄 Learning affine warp parameters...")
print(f"   Warmup (no mask): iterations 1-{warmup_iterations}")
print(f"   With 95th percentile mask: iterations {warmup_iterations+1}-{n_iterations}")
print(f"\n{'Iter':>6} {'Loss':>12} {'Masked':>8} {'a':>8} {'b':>8} {'c':>8} {'d':>8} {'tx':>8} {'ty':>8}")
print("-" * 90)

for i in range(n_iterations):
    optimizer.zero_grad()
    
    # Forward pass: warp x_prev
    warped = warp_model(x_prev_t)
    
    # Determine whether to use mask (after warmup)
    use_mask = i >= warmup_iterations
    
    # Compute masked loss
    loss, mask = masked_loss(warped, x_target_t, use_mask=use_mask, percentile=95)
    
    # Add regularization term
    loss = loss + torch.log(1e-7 + torch.norm(warped - x_target_t))
    
    # Backward pass
    loss.backward()
    optimizer.step()
    
    # Record
    losses.append(loss.item())
    mask_history.append(use_mask)
    for name in param_history:
        param_history[name].append(getattr(warp_model, name).item())
    
    # Print progress
    if (i + 1) % 50 == 0 or i == 0:
        mask_str = "Yes" if use_mask else "No"
        print(f"{i+1:6d} {loss.item():12.8f} {mask_str:>8} "
              f"{warp_model.a.item():8.5f} {warp_model.b.item():8.5f} "
              f"{warp_model.c.item():8.5f} {warp_model.d.item():8.5f} "
              f"{warp_model.tx.item():8.5f} {warp_model.ty.item():8.5f}")

print("\n✅ Training complete!")

# ---- Final evaluation ----
warp_model.eval()
with torch.no_grad():
    warped_final = warp_model(x_prev_t)
    final_mse = F.mse_loss(warped_final, x_target_t).item()
    
    # Get final mask for visualization
    _, final_mask = masked_loss(warped_final, x_target_t, use_mask=True, percentile=95)
    
# Baseline: no transformation
baseline_mse = F.mse_loss(x_prev_t, x_target_t).item()

final_psnr = 10 * np.log10(1.0 / max(final_mse, 1e-10))
baseline_psnr = 10 * np.log10(1.0 / max(baseline_mse, 1e-10))

# Compute masked PSNR (only on non-outlier pixels)
with torch.no_grad():
    diff_masked = (warped_final - x_target_t) * final_mask.float()
    mse_masked = (diff_masked ** 2).sum() / final_mask.sum()
    psnr_masked = 10 * np.log10(1.0 / max(mse_masked.item(), 1e-10))

print(f"\n📊 Results:")
print(f"   Baseline (no warp):    MSE={baseline_mse:.8f}  PSNR={baseline_psnr:.2f} dB")
print(f"   Learned affine warp:   MSE={final_mse:.8f}  PSNR={final_psnr:.2f} dB")
print(f"   Masked PSNR (95%):     PSNR={psnr_masked:.2f} dB")
print(f"   Improvement: {final_psnr - baseline_psnr:.2f} dB")
print(f"\n   Pixels masked out (top 5%): {(~final_mask).sum().item()} / {final_mask.numel()}")

# Print learned parameters
theta = warp_model.get_theta()
print(f"\n📐 Learned affine matrix:")
print(f"   [[{theta[0,0,0].item():.6f}, {theta[0,0,1].item():.6f}, {theta[0,0,2].item():.6f}],")
print(f"    [{theta[0,1,0].item():.6f}, {theta[0,1,1].item():.6f}, {theta[0,1,2].item():.6f}]]")

In [ ]:
# ---- Visualization ----
fig, axes = plt.subplots(2, 4, figsize=(20, 10))

# Top row: images
warped_np = warped_final[0, 0].cpu().numpy()

axes[0, 0].imshow(x_prev, vmin=0, vmax=1)
axes[0, 0].set_title("x_prev (Input)")
axes[0, 0].axis("off")

axes[0, 1].imshow(warped_np, vmin=0, vmax=1)
axes[0, 1].set_title(f"Warped x_prev\n(Learned Affine)")
axes[0, 1].axis("off")

axes[0, 2].imshow(x_target, vmin=0, vmax=1)
axes[0, 2].set_title("x_target (Ground Truth)")
axes[0, 2].axis("off")

# Difference: warped vs target
diff = warped_np - x_target
vmax = max(np.abs(diff).max(), 1e-6)
im = axes[0, 3].imshow(diff, cmap='RdBu', vmin=-vmax, vmax=vmax)
axes[0, 3].set_title(f"Warped − Target\nPSNR: {final_psnr:.2f} dB")
axes[0, 3].axis("off")
plt.colorbar(im, ax=axes[0, 3], fraction=0.046)

# Bottom row: training curves and comparison
# Loss curve
axes[1, 0].semilogy(losses)
axes[1, 0].set_xlabel("Iteration")
axes[1, 0].set_ylabel("MSE Loss (log scale)")
axes[1, 0].set_title("Training Loss")
axes[1, 0].grid(True, alpha=0.3)

# Parameter evolution
axes[1, 1].plot(param_history['tx'], label='tx')
axes[1, 1].plot(param_history['ty'], label='ty')
axes[1, 1].set_xlabel("Iteration")
axes[1, 1].set_ylabel("Value")
axes[1, 1].set_title("Translation Parameters")
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

axes[1, 2].plot(param_history['a'], label='a (scale_x-1)')
axes[1, 2].plot(param_history['d'], label='d (scale_y-1)')
axes[1, 2].plot(param_history['b'], label='b (shear_x)')
axes[1, 2].plot(param_history['c'], label='c (shear_y)')
axes[1, 2].set_xlabel("Iteration")
axes[1, 2].set_ylabel("Value")
axes[1, 2].set_title("Scale/Shear Parameters")
axes[1, 2].legend()
axes[1, 2].grid(True, alpha=0.3)

# Baseline difference for comparison
diff_baseline = x_prev - x_target
vmax_bl = max(np.abs(diff_baseline).max(), 1e-6)
im2 = axes[1, 3].imshow(diff_baseline, cmap='RdBu', vmin=-vmax_bl, vmax=vmax_bl)
axes[1, 3].set_title(f"x_prev − Target (No Warp)\nPSNR: {baseline_psnr:.2f} dB")
axes[1, 3].axis("off")
plt.colorbar(im2, ax=axes[1, 3], fraction=0.046)

plt.suptitle("Learnable Affine Warping: x_prev → x_target", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

# Print summary
print("\n" + "="*60)
print("SUMMARY: Learnable Affine Warp")
print("="*60)
print(f"Input:  x_prev (frame {idx_start})")
print(f"Target: x_target (frame {idx_start + 1})")
print(f"\nBaseline PSNR (no warp):  {baseline_psnr:.2f} dB")
print(f"Affine warp PSNR:         {final_psnr:.2f} dB")
print(f"Improvement:              +{final_psnr - baseline_psnr:.2f} dB")
print(f"\nLearned transformation (6 parameters):")
print(f"  Scale X:     {1 + warp_model.a.item():.6f}")
print(f"  Scale Y:     {1 + warp_model.d.item():.6f}")
print(f"  Shear X:     {warp_model.b.item():.6f}")
print(f"  Shear Y:     {warp_model.c.item():.6f}")
print(f"  Translate X: {warp_model.tx.item():.6f} (normalized)")
print(f"  Translate Y: {warp_model.ty.item():.6f} (normalized)")
print("="*60)

In [ ]:
im_target = plt.imshow(dataset.processors[3]._flat_first)
plt.colorbar(im_target, fraction=0.01)

In [ ]:
im_target = plt.imshow(diff, cmap='bwr', vmin=-0.07, vmax=0.07)
plt.colorbar(im_target, fraction=0.01)

### Compare with Drift Predictor Methods

Let's compare the learned affine warp against the drift-based methods on the same frame pair.

In [ ]:
# Compare learned affine warp with drift predictor methods
from compress_ct.drift_predictor import DriftPredictor

comparison_results = {}

# Baseline (no prediction)
comparison_results["No Warp"] = {
    "pred": x_prev,
    "mse": baseline_mse,
    "psnr": baseline_psnr
}

# Learned affine warp
comparison_results["Learned Affine"] = {
    "pred": warped_np,
    "mse": final_mse,
    "psnr": final_psnr
}

# Drift predictor methods (using 2 frames: x_prev_prev and x_prev to predict x_target)
# For fair comparison with affine, we'll use just x_prev to predict x_target
# But drift predictor needs 2 frames, so let's get x_prev_prev
x_prev_prev = processor.get_projection(idx_start - 1, normalize=True)

for mode in ["phase_correlation", "optical_flow", "cross_correlation"]:
    dp = DriftPredictor(mode=mode, num_input_projections=2, verbose=False)
    pred = dp.predict_frame([x_prev_prev, x_prev])
    mse = np.mean((pred - x_target) ** 2)
    psnr = 10 * np.log10(1.0 / max(mse, 1e-10))
    comparison_results[f"Drift ({mode})"] = {
        "pred": pred,
        "mse": mse,
        "psnr": psnr
    }

# Print comparison table
print("="*70)
print(f"{'Method':<30} {'MSE':>15} {'PSNR (dB)':>15}")
print("-"*70)
for name, res in comparison_results.items():
    print(f"{name:<30} {res['mse']:>15.8f} {res['psnr']:>15.2f}")
print("="*70)

# Find best method
best = max(comparison_results.items(), key=lambda x: x[1]['psnr'])
print(f"\n🏆 Best method: {best[0]} with PSNR {best[1]['psnr']:.2f} dB")

In [ ]:
# Visual comparison of all methods
n_methods = len(comparison_results)
fig, axes = plt.subplots(2, n_methods, figsize=(5 * n_methods, 10))

for col, (name, res) in enumerate(comparison_results.items()):
    # Top row: predictions
    axes[0, col].imshow(res["pred"], vmin=0, vmax=1)
    axes[0, col].set_title(f"{name}\nPSNR: {res['psnr']:.2f} dB")
    axes[0, col].axis("off")
    
    # Bottom row: error maps
    diff = res["pred"] - x_target
    vmax = max(np.abs(diff).max(), 1e-6)
    im = axes[1, col].imshow(diff, cmap='RdBu', vmin=-vmax, vmax=vmax)
    axes[1, col].set_title(f"Error: {name}")
    axes[1, col].axis("off")
    plt.colorbar(im, ax=axes[1, col], fraction=0.046)

plt.suptitle(f"Frame Prediction Comparison (Target: frame {idx_start + 1})", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

### Test on Multiple Frame Pairs

Let's evaluate the learned affine approach on several consecutive frame pairs to see how consistent it is.

In [ ]:
def learn_affine_warp(x_prev, x_target, n_iterations=300, lr=0.01, device='cuda', verbose=False):
    """
    Learn the affine transformation that best warps x_prev into x_target.
    
    Returns:
        warped: The warped x_prev
        theta: The learned 2x3 affine matrix
        params: Dictionary of learned parameters
        final_loss: Final MSE loss
    """
    x_prev_t = torch.tensor(x_prev, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device)
    x_target_t = torch.tensor(x_target, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device)
    
    model = LearnableAffineWarp().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    for i in range(n_iterations):
        optimizer.zero_grad()
        warped = model(x_prev_t)
        loss = F.mse_loss(warped, x_target_t)
        loss.backward()
        optimizer.step()
        
        if verbose and (i + 1) % 100 == 0:
            print(f"  Iter {i+1}: loss={loss.item():.8f}")
    
    model.eval()
    with torch.no_grad():
        warped_final = model(x_prev_t)
        final_loss = F.mse_loss(warped_final, x_target_t).item()
    
    theta = model.get_theta()
    params = {
        'scale_x': 1 + model.a.item(),
        'scale_y': 1 + model.d.item(),
        'shear_x': model.b.item(),
        'shear_y': model.c.item(),
        'tx': model.tx.item(),
        'ty': model.ty.item(),
    }
    
    return warped_final[0, 0].cpu().detach().numpy(), theta.cpu().detach().numpy(), params, final_loss


# Test on multiple frame pairs
test_indices = list(range(20, 28))  # 8 frame pairs
multi_results = []

print("🔄 Learning affine warps for multiple frame pairs...")
print(f"{'Pair':>6} {'Baseline PSNR':>14} {'Affine PSNR':>14} {'Improvement':>12} {'tx':>10} {'ty':>10}")
print("-" * 70)

for idx in test_indices:
    xp = processor.get_projection(idx, normalize=True)
    xt = processor.get_projection(idx + 1, normalize=True)
    
    # Learn the warp
    warped, theta, params, loss = learn_affine_warp(xp, xt, n_iterations=300, lr=0.01, device=device)
    
    # Compute metrics
    baseline_mse = np.mean((xp - xt) ** 2)
    affine_mse = loss
    
    baseline_psnr = 10 * np.log10(1.0 / max(baseline_mse, 1e-10))
    affine_psnr = 10 * np.log10(1.0 / max(affine_mse, 1e-10))
    improvement = affine_psnr - baseline_psnr
    
    multi_results.append({
        'idx': idx,
        'baseline_psnr': baseline_psnr,
        'affine_psnr': affine_psnr,
        'improvement': improvement,
        'params': params,
        'warped': warped,
        'target': xt,
    })
    
    print(f"{idx}->{idx+1}  {baseline_psnr:14.2f} {affine_psnr:14.2f} {improvement:+12.2f} {params['tx']:10.6f} {params['ty']:10.6f}")

# Summary statistics
avg_baseline = np.mean([r['baseline_psnr'] for r in multi_results])
avg_affine = np.mean([r['affine_psnr'] for r in multi_results])
avg_improvement = np.mean([r['improvement'] for r in multi_results])

print("-" * 70)
print(f"{'Avg':>6} {avg_baseline:14.2f} {avg_affine:14.2f} {avg_improvement:+12.2f}")
print(f"\n✅ Average improvement: +{avg_improvement:.2f} dB")

In [ ]:
# Visualize results from multiple frame pairs
n_show = min(4, len(multi_results))
fig, axes = plt.subplots(3, n_show, figsize=(5 * n_show, 14))

for col in range(n_show):
    r = multi_results[col]
    xp = processor.get_projection(r['idx'], normalize=True)
    
    # Row 0: Original x_prev
    axes[0, col].imshow(xp, vmin=0, vmax=1)
    axes[0, col].set_title(f"Frame {r['idx']} (Input)")
    axes[0, col].axis("off")
    
    # Row 1: Warped result
    axes[1, col].imshow(r['warped'], vmin=0, vmax=1)
    axes[1, col].set_title(f"Warped → Frame {r['idx']+1}\nPSNR: {r['affine_psnr']:.2f} dB")
    axes[1, col].axis("off")
    
    # Row 2: Error map
    diff = r['warped'] - r['target']
    vmax = max(np.abs(diff).max(), 1e-6)
    im = axes[2, col].imshow(diff, cmap='RdBu', vmin=-vmax, vmax=vmax)
    axes[2, col].set_title(f"Error (Warped − Target)\nImproved: +{r['improvement']:.2f} dB")
    axes[2, col].axis("off")
    plt.colorbar(im, ax=axes[2, col], fraction=0.046)

plt.suptitle("Learned Affine Warp: Multiple Frame Pairs", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

# Plot learned translation parameters over time
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

frame_labels = [f"{r['idx']}->{r['idx']+1}" for r in multi_results]
txs = [r['params']['tx'] for r in multi_results]
tys = [r['params']['ty'] for r in multi_results]

axes[0].bar(range(len(multi_results)), txs, alpha=0.7, label='tx')
axes[0].bar(range(len(multi_results)), tys, alpha=0.7, label='ty')
axes[0].set_xticks(range(len(multi_results)))
axes[0].set_xticklabels(frame_labels, rotation=45)
axes[0].set_ylabel("Translation (normalized)")
axes[0].set_title("Learned Translation per Frame Pair")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

improvements = [r['improvement'] for r in multi_results]
axes[1].bar(range(len(multi_results)), improvements, color='green', alpha=0.7)
axes[1].set_xticks(range(len(multi_results)))
axes[1].set_xticklabels(frame_labels, rotation=45)
axes[1].set_ylabel("PSNR Improvement (dB)")
axes[1].set_title("Quality Improvement from Affine Warp")
axes[1].axhline(y=avg_improvement, color='red', linestyle='--', label=f'Avg: +{avg_improvement:.2f} dB')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Compression Implications

The learned affine warp captures the geometric transformation between frames.
For compression, we could:

1. **Store the 6 affine parameters** (24 bytes as float32) instead of dense prediction
2. **Encode residuals** = target - warped_prediction
3. **Residuals should have lower variance** than raw frame differences

This is a form of **motion-compensated compression** where the "motion" is an affine transform.

In [ ]:
# Analyze residual statistics for compression
print("📊 Residual Analysis for Compression")
print("="*70)
print(f"{'Frame Pair':>12} {'Raw Diff Std':>14} {'Residual Std':>14} {'Reduction':>12}")
print("-"*70)

residual_stats = []
for r in multi_results:
    xp = processor.get_projection(r['idx'], normalize=True)
    
    # Raw difference (no prediction)
    raw_diff = r['target'] - xp
    raw_std = np.std(raw_diff)
    
    # Residual after affine warp
    residual = r['target'] - r['warped']
    res_std = np.std(residual)
    
    reduction = (1 - res_std / raw_std) * 100
    
    residual_stats.append({
        'raw_std': raw_std,
        'res_std': res_std,
        'reduction': reduction,
    })
    
    print(f"{r['idx']}->{r['idx']+1}  {raw_std:14.6f} {res_std:14.6f} {reduction:11.1f}%")

avg_raw_std = np.mean([s['raw_std'] for s in residual_stats])
avg_res_std = np.mean([s['res_std'] for s in residual_stats])
avg_reduction = np.mean([s['reduction'] for s in residual_stats])

print("-"*70)
print(f"{'Average':>12} {avg_raw_std:14.6f} {avg_res_std:14.6f} {avg_reduction:11.1f}%")
print("="*70)

print(f"\n💡 Key insight: Affine warping reduces residual std by ~{avg_reduction:.1f}%")
print(f"   This means the residuals are easier to compress!")
print(f"\n📦 Storage cost per frame:")
print(f"   - 6 affine parameters: 24 bytes (float32)")
print(f"   - Residual with lower variance → better compression")

# Histogram comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pick one frame pair for histogram
r = multi_results[0]
xp = processor.get_projection(r['idx'], normalize=True)
raw_diff = r['target'] - xp
residual = r['target'] - r['warped']

axes[0].hist(raw_diff.flatten(), bins=100, alpha=0.7, label=f'Raw Diff (std={np.std(raw_diff):.4f})', density=True)
axes[0].hist(residual.flatten(), bins=100, alpha=0.7, label=f'After Warp (std={np.std(residual):.4f})', density=True)
axes[0].set_xlabel("Pixel Difference")
axes[0].set_ylabel("Density")
axes[0].set_title(f"Residual Distribution (Frame {r['idx']}->{r['idx']+1})")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Cumulative distribution
axes[1].hist(np.abs(raw_diff.flatten()), bins=100, alpha=0.7, cumulative=True, density=True, 
             label='|Raw Diff|', histtype='step', linewidth=2)
axes[1].hist(np.abs(residual.flatten()), bins=100, alpha=0.7, cumulative=True, density=True,
             label='|After Warp|', histtype='step', linewidth=2)
axes[1].set_xlabel("|Pixel Difference|")
axes[1].set_ylabel("Cumulative Density")
axes[1].set_title("Cumulative Distribution of Absolute Errors")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 18. Per-Patch Affine Fitting

Instead of learning a single global affine transformation for the entire frame, we can learn
**separate affine transforms per patch**. This allows for more fine-grained motion compensation,
capturing local deformations that a single global affine cannot model.

For compression, we store:
- One 6-parameter affine transform per patch (24 bytes × num_patches)
- The residual (target_patch - warped_patch) which should have lower variance

We'll test this approach using samples from Section 10.

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from typing import Tuple, Dict, List

class PatchAffineWarp(torch.nn.Module):
    """
    Learns an affine transformation for a single patch.
    Parameterized as offsets from identity transform.
    """
    def __init__(self):
        super().__init__()
        self.a = torch.nn.Parameter(torch.zeros(1))   # scale_x - 1
        self.b = torch.nn.Parameter(torch.zeros(1))   # shear_x
        self.c = torch.nn.Parameter(torch.zeros(1))   # shear_y  
        self.d = torch.nn.Parameter(torch.zeros(1))   # scale_y - 1
        self.tx = torch.nn.Parameter(torch.zeros(1))  # translate_x
        self.ty = torch.nn.Parameter(torch.zeros(1))  # translate_y
    
    def get_theta(self):
        """Build the 2x3 affine matrix."""
        theta = torch.zeros(1, 2, 3, device=self.a.device)
        theta[0, 0, 0] = 1 + self.a
        theta[0, 0, 1] = self.b
        theta[0, 0, 2] = self.tx
        theta[0, 1, 0] = self.c
        theta[0, 1, 1] = 1 + self.d
        theta[0, 1, 2] = self.ty
        return theta
    
    def forward(self, x):
        """Warp input x using learned affine transform."""
        theta = self.get_theta()
        grid = F.affine_grid(theta, x.size(), align_corners=False)
        warped = F.grid_sample(x, grid, mode='bilinear', padding_mode='border', align_corners=False)
        return warped
    
    def get_params_dict(self):
        """Return learned parameters as a dictionary."""
        return {
            'scale_x': 1 + self.a.item(),
            'scale_y': 1 + self.d.item(),
            'shear_x': self.b.item(),
            'shear_y': self.c.item(),
            'tx': self.tx.item(),
            'ty': self.ty.item(),
        }


def fit_affine_to_patch(
    x_prev_patch: np.ndarray,
    x_target_patch: np.ndarray,
    n_iterations: int = 200,
    lr: float = 0.01,
    device: str = 'cuda'
) -> Tuple[np.ndarray, Dict, float]:
    """
    Learn the optimal affine transform for a single patch.
    
    Args:
        x_prev_patch: Source patch (H, W)
        x_target_patch: Target patch (H, W)
        n_iterations: Number of optimization steps
        lr: Learning rate
        device: Device to use
        
    Returns:
        warped: Warped patch
        params: Dictionary of affine parameters
        final_loss: Final MSE loss
    """
    x_prev_t = torch.tensor(x_prev_patch, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device)
    x_target_t = torch.tensor(x_target_patch, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device)
    
    model = PatchAffineWarp().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    for _ in range(n_iterations):
        optimizer.zero_grad()
        warped = model(x_prev_t)
        loss = F.mse_loss(warped, x_target_t)
        loss.backward()
        optimizer.step()
    
    model.eval()
    with torch.no_grad():
        warped_final = model(x_prev_t)
        final_loss = F.mse_loss(warped_final, x_target_t).item()
    
    return warped_final[0, 0].cpu().numpy(), model.get_params_dict(), final_loss


def per_patch_affine_fitting(
    x_prev: np.ndarray,
    x_target: np.ndarray,
    patch_size: int = 128,
    overlap: int = 0,
    n_iterations: int = 200,
    lr: float = 0.01,
    device: str = 'cuda',
    verbose: bool = True
) -> Dict:
    """
    Perform per-patch affine fitting on the entire frame.
    
    Args:
        x_prev: Source frame (H, W)
        x_target: Target frame (H, W)
        patch_size: Size of each patch
        overlap: Overlap between patches (for blending)
        n_iterations: Optimization iterations per patch
        lr: Learning rate
        device: Device
        verbose: Print progress
        
    Returns:
        Dictionary containing:
        - warped_frame: Full reconstructed frame
        - patch_params: List of per-patch affine parameters
        - patch_locations: List of (y, x) patch locations
        - metrics: PSNR and MSE values
    """
    H, W = x_prev.shape
    stride = patch_size - overlap
    
    # Calculate number of patches
    n_patches_h = (H - overlap) // stride
    n_patches_w = (W - overlap) // stride
    
    # Initialize output arrays
    warped_frame = np.zeros_like(x_prev)
    weight_map = np.zeros_like(x_prev)  # For blending overlapping regions
    
    patch_params = []
    patch_locations = []
    patch_losses = []
    
    total_patches = n_patches_h * n_patches_w
    patch_idx = 0
    
    if verbose:
        print(f"🔄 Per-patch affine fitting: {n_patches_h}×{n_patches_w} = {total_patches} patches")
        print(f"   Patch size: {patch_size}, Overlap: {overlap}, Stride: {stride}")
    
    for i in range(n_patches_h):
        for j in range(n_patches_w):
            y_start = i * stride
            x_start = j * stride
            y_end = min(y_start + patch_size, H)
            x_end = min(x_start + patch_size, W)
            
            # Handle edge patches - ensure we get a full patch_size x patch_size region
            if y_end - y_start < patch_size:
                y_start = H - patch_size
                y_end = H
            if x_end - x_start < patch_size:
                x_start = W - patch_size
                x_end = W
            
            # Extract patches
            prev_patch = x_prev[y_start:y_end, x_start:x_end]
            target_patch = x_target[y_start:y_end, x_start:x_end]
            
            # Fit affine transform
            warped_patch, params, loss = fit_affine_to_patch(
                prev_patch, target_patch, n_iterations, lr, device
            )
            
            # Store results
            patch_params.append(params)
            patch_locations.append((y_start, x_start))
            patch_losses.append(loss)
            
            # Add to output (simple averaging for overlapping regions)
            warped_frame[y_start:y_end, x_start:x_end] += warped_patch
            weight_map[y_start:y_end, x_start:x_end] += 1
            
            patch_idx += 1
            if verbose and (patch_idx % 10 == 0 or patch_idx == total_patches):
                print(f"   Processed {patch_idx}/{total_patches} patches...")
    
    # Normalize overlapping regions
    weight_map[weight_map == 0] = 1  # Avoid division by zero
    warped_frame = warped_frame / weight_map
    
    # Compute metrics
    mse = np.mean((warped_frame - x_target) ** 2)
    psnr = 10 * np.log10(1.0 / max(mse, 1e-10))
    
    baseline_mse = np.mean((x_prev - x_target) ** 2)
    baseline_psnr = 10 * np.log10(1.0 / max(baseline_mse, 1e-10))
    
    return {
        'warped_frame': warped_frame,
        'patch_params': patch_params,
        'patch_locations': patch_locations,
        'patch_losses': patch_losses,
        'n_patches': total_patches,
        'metrics': {
            'mse': mse,
            'psnr': psnr,
            'baseline_mse': baseline_mse,
            'baseline_psnr': baseline_psnr,
            'improvement': psnr - baseline_psnr,
        }
    }


print("✅ Per-patch affine fitting functions defined!")

### Test Per-Patch Affine Fitting on Section 10 Sample

Let's test the per-patch affine approach using the same sample from Section 10 (`dataset[7000]`).

In [ ]:
# Use the same sample from Section 10
sample = dataset[7000]
x_prev = sample['input'][-1].numpy()  # Last input projection (P_{i+k-1})
x_target = sample['target'][0].numpy()  # Target projection (P_{i+k})

print(f"📊 Sample from Section 10:")
print(f"   Shape: {x_prev.shape}")
print(f"   x_prev range: [{x_prev.min():.4f}, {x_prev.max():.4f}]")
print(f"   x_target range: [{x_target.min():.4f}, {x_target.max():.4f}]")

# Test different patch sizes
PATCH_SIZES = [64, 128, 256]
results_by_patch_size = {}

for patch_size in PATCH_SIZES:
    print(f"\n{'='*70}")
    print(f"📐 Testing patch_size={patch_size}")
    print(f"{'='*70}")
    
    result = per_patch_affine_fitting(
        x_prev, x_target,
        patch_size=patch_size,
        overlap=0,
        n_iterations=200,
        lr=0.02,
        device=device,
        verbose=True
    )
    
    results_by_patch_size[patch_size] = result
    
    m = result['metrics']
    print(f"\n📈 Results for patch_size={patch_size}:")
    print(f"   Baseline PSNR: {m['baseline_psnr']:.2f} dB")
    print(f"   Per-Patch Affine PSNR: {m['psnr']:.2f} dB")
    print(f"   Improvement: +{m['improvement']:.2f} dB")
    print(f"   Number of patches: {result['n_patches']}")
    print(f"   Storage: {result['n_patches'] * 6 * 4} bytes (6 floats × 4 bytes × {result['n_patches']} patches)")

# Summary table
print(f"\n{'='*70}")
print("📊 SUMMARY: Per-Patch Affine Fitting")
print(f"{'='*70}")
print(f"{'Patch Size':>12} {'Num Patches':>12} {'PSNR (dB)':>12} {'Improvement':>12} {'Storage (KB)':>14}")
print("-"*70)
for ps in PATCH_SIZES:
    r = results_by_patch_size[ps]
    m = r['metrics']
    storage_kb = (r['n_patches'] * 6 * 4) / 1024
    print(f"{ps:>12} {r['n_patches']:>12} {m['psnr']:>12.2f} {m['improvement']:>+12.2f} {storage_kb:>14.2f}")
print(f"{'Baseline':>12} {'-':>12} {m['baseline_psnr']:>12.2f} {0.0:>+12.2f} {0.0:>14.2f}")
print("-"*70)

In [ ]:
# Visualization: Compare different patch sizes
fig, axes = plt.subplots(3, len(PATCH_SIZES) + 2, figsize=(5 * (len(PATCH_SIZES) + 2), 14))

# Row 0: Input, Target, Global baseline
axes[0, 0].imshow(x_prev, vmin=0, vmax=1)
axes[0, 0].set_title("x_prev (Input)")
axes[0, 0].axis("off")

axes[0, 1].imshow(x_target, vmin=0, vmax=1)
axes[0, 1].set_title("x_target (Ground Truth)")
axes[0, 1].axis("off")

for col, ps in enumerate(PATCH_SIZES):
    r = results_by_patch_size[ps]
    axes[0, col + 2].imshow(r['warped_frame'], vmin=0, vmax=1)
    axes[0, col + 2].set_title(f"Patch {ps}×{ps}\nPSNR: {r['metrics']['psnr']:.2f} dB")
    axes[0, col + 2].axis("off")

# Row 1: Difference maps
diff_baseline = x_prev - x_target
vmax_bl = max(np.abs(diff_baseline).max(), 1e-6)
im = axes[1, 0].imshow(diff_baseline, cmap='RdBu', vmin=-vmax_bl, vmax=vmax_bl)
axes[1, 0].set_title(f"Baseline Error\n(x_prev - x_target)")
axes[1, 0].axis("off")
plt.colorbar(im, ax=axes[1, 0], fraction=0.046)

axes[1, 1].axis("off")  # Empty cell

for col, ps in enumerate(PATCH_SIZES):
    r = results_by_patch_size[ps]
    diff = r['warped_frame'] - x_target
    vmax = max(np.abs(diff).max(), 1e-6)
    im = axes[1, col + 2].imshow(diff, cmap='RdBu', vmin=-vmax_bl, vmax=vmax_bl)
    axes[1, col + 2].set_title(f"Error (patch {ps})\n+{r['metrics']['improvement']:.2f} dB improvement")
    axes[1, col + 2].axis("off")
    plt.colorbar(im, ax=axes[1, col + 2], fraction=0.046)

# Row 2: Per-patch parameter visualization (translation magnitude)
axes[2, 0].axis("off")
axes[2, 1].axis("off")

for col, ps in enumerate(PATCH_SIZES):
    r = results_by_patch_size[ps]
    # Create a visualization of translation magnitudes
    H, W = x_prev.shape
    tx_map = np.zeros((H, W))
    ty_map = np.zeros((H, W))
    
    for (y, x), params in zip(r['patch_locations'], r['patch_params']):
        tx_map[y:y+ps, x:x+ps] = params['tx']
        ty_map[y:y+ps, x:x+ps] = params['ty']
    
    # Show translation magnitude
    trans_mag = np.sqrt(tx_map**2 + ty_map**2)
    im = axes[2, col + 2].imshow(trans_mag, cmap='viridis')
    axes[2, col + 2].set_title(f"Translation Magnitude\n(patch {ps})")
    axes[2, col + 2].axis("off")
    plt.colorbar(im, ax=axes[2, col + 2], fraction=0.046)

plt.suptitle("Per-Patch Affine Fitting: Comparison Across Patch Sizes", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Compare with global affine and drift methods
from compress_ct.drift_predictor import DriftPredictor

comparison_results = {}

# 1. Baseline (no prediction)
baseline_mse = np.mean((x_prev - x_target) ** 2)
baseline_psnr = 10 * np.log10(1.0 / max(baseline_mse, 1e-10))
comparison_results["No Prediction"] = {
    "pred": x_prev,
    "mse": baseline_mse,
    "psnr": baseline_psnr,
    "storage_bytes": 0
}

# 2. Global Affine (from Section 17)
print("🔄 Learning global affine transform...")
x_prev_t = torch.tensor(x_prev, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device)
x_target_t = torch.tensor(x_target, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device)

global_warp = LearnableAffineWarp().to(device)
optimizer = torch.optim.Adam(global_warp.parameters(), lr=0.01)

for _ in range(500):
    optimizer.zero_grad()
    warped = global_warp(x_prev_t)
    loss = F.mse_loss(warped, x_target_t)
    loss.backward()
    optimizer.step()

global_warp.eval()
with torch.no_grad():
    global_warped = global_warp(x_prev_t)[0, 0].cpu().numpy()
    global_mse = np.mean((global_warped - x_target) ** 2)
    global_psnr = 10 * np.log10(1.0 / max(global_mse, 1e-10))

comparison_results["Global Affine"] = {
    "pred": global_warped,
    "mse": global_mse,
    "psnr": global_psnr,
    "storage_bytes": 6 * 4  # 6 float32 params
}
print(f"   Global Affine PSNR: {global_psnr:.2f} dB")

# 3. Per-Patch Affine (best patch size)
best_ps = max(results_by_patch_size.keys(), key=lambda ps: results_by_patch_size[ps]['metrics']['psnr'])
best_result = results_by_patch_size[best_ps]
comparison_results[f"Per-Patch Affine ({best_ps})"] = {
    "pred": best_result['warped_frame'],
    "mse": best_result['metrics']['mse'],
    "psnr": best_result['metrics']['psnr'],
    "storage_bytes": best_result['n_patches'] * 6 * 4
}

# Add other patch sizes
for ps in PATCH_SIZES:
    if ps != best_ps:
        r = results_by_patch_size[ps]
        comparison_results[f"Per-Patch Affine ({ps})"] = {
            "pred": r['warped_frame'],
            "mse": r['metrics']['mse'],
            "psnr": r['metrics']['psnr'],
            "storage_bytes": r['n_patches'] * 6 * 4
        }

# 4. Drift Predictor Methods (need 2 frames)
# We'll use the first 2 input projections from the sample
x_prev_prev = sample['input'][0].numpy()
x_prev_for_drift = sample['input'][-1].numpy()

for mode in ["phase_correlation", "optical_flow", "cross_correlation"]:
    dp = DriftPredictor(mode=mode, num_input_projections=NUM_INPUT_PROJECTIONS, verbose=False)
    prev_frames = [sample['input'][i].numpy() for i in range(NUM_INPUT_PROJECTIONS)]
    drift_pred_frame = dp.predict_frame(prev_frames)
    drift_mse = np.mean((drift_pred_frame - x_target) ** 2)
    drift_psnr = 10 * np.log10(1.0 / max(drift_mse, 1e-10))
    comparison_results[f"Drift ({mode})"] = {
        "pred": drift_pred_frame,
        "mse": drift_mse,
        "psnr": drift_psnr,
        "storage_bytes": 16  # 2 float32 for (dx, dy)
    }

# Print comparison table
print("\n" + "="*90)
print("📊 COMPREHENSIVE COMPARISON: Per-Patch vs Global Affine vs Drift Methods")
print("="*90)
print(f"{'Method':<30} {'PSNR (dB)':>12} {'Improvement':>12} {'Storage':>15}")
print("-"*90)

# Sort by PSNR
sorted_results = sorted(comparison_results.items(), key=lambda x: x[1]['psnr'], reverse=True)

for name, res in sorted_results:
    improvement = res['psnr'] - baseline_psnr
    storage_str = f"{res['storage_bytes']} bytes" if res['storage_bytes'] < 1024 else f"{res['storage_bytes']/1024:.1f} KB"
    print(f"{name:<30} {res['psnr']:>12.2f} {improvement:>+12.2f} {storage_str:>15}")

print("="*90)
print(f"\n🏆 Best method: {sorted_results[0][0]} with PSNR {sorted_results[0][1]['psnr']:.2f} dB")

In [ ]:
# Visual comparison of all methods
methods_to_show = ["No Prediction", "Global Affine", f"Per-Patch Affine ({best_ps})", "Drift (phase_correlation)"]
n_methods = len(methods_to_show)

fig, axes = plt.subplots(2, n_methods + 1, figsize=(5 * (n_methods + 1), 10))

# Ground truth in first column
axes[0, 0].imshow(x_target, vmin=0, vmax=1)
axes[0, 0].set_title("Ground Truth\n(x_target)")
axes[0, 0].axis("off")
axes[1, 0].axis("off")

# Each method
for col, name in enumerate(methods_to_show, 1):
    res = comparison_results[name]
    
    # Prediction
    axes[0, col].imshow(res['pred'], vmin=0, vmax=1)
    axes[0, col].set_title(f"{name}\nPSNR: {res['psnr']:.2f} dB")
    axes[0, col].axis("off")
    
    # Error map
    diff = res['pred'] - x_target
    vmax = max(np.abs(diff).max(), 1e-6)
    im = axes[1, col].imshow(diff, cmap='RdBu', vmin=-vmax, vmax=vmax)
    axes[1, col].set_title(f"Error: {name}")
    axes[1, col].axis("off")
    plt.colorbar(im, ax=axes[1, col], fraction=0.046)

plt.suptitle("Per-Patch Affine Fitting vs Other Methods", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

### Residual Statistics for Compression

For compression purposes, we care about the residual variance. Lower residual variance means better compressibility.

In [ ]:
# Analyze residual statistics for compression
print("📊 Residual Analysis for Compression")
print("="*90)
print(f"{'Method':<30} {'Residual Std':>12} {'Residual Max':>12} {'Reduction %':>12} {'Entropy Est':>12}")
print("-"*90)

residual_stats = {}
baseline_residual = x_target - x_prev
baseline_std = np.std(baseline_residual)

for name, res in sorted_results:
    residual = x_target - res['pred']
    res_std = np.std(residual)
    res_max = np.abs(residual).max()
    reduction = (1 - res_std / baseline_std) * 100 if name != "No Prediction" else 0.0
    
    # Simple entropy estimate based on histogram (bits per pixel)
    hist, _ = np.histogram(residual.flatten(), bins=256, range=(-1, 1))
    hist = hist / hist.sum()  # normalize
    hist = hist[hist > 0]  # remove zeros
    entropy = -np.sum(hist * np.log2(hist))
    
    residual_stats[name] = {
        'std': res_std,
        'max': res_max,
        'reduction': reduction,
        'entropy': entropy,
        'residual': residual
    }
    
    print(f"{name:<30} {res_std:>12.6f} {res_max:>12.6f} {reduction:>+11.1f}% {entropy:>12.2f}")

print("="*90)

# Histogram comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Compare residual distributions
methods_for_hist = ["No Prediction", "Global Affine", f"Per-Patch Affine ({best_ps})"]
colors = ['red', 'blue', 'green']

for ax_idx, (name, color) in enumerate(zip(methods_for_hist, colors)):
    rs = residual_stats[name]
    axes[0].hist(rs['residual'].flatten(), bins=100, alpha=0.5, label=f'{name} (std={rs["std"]:.4f})', 
                 color=color, density=True)

axes[0].set_xlabel("Residual Value")
axes[0].set_ylabel("Density")
axes[0].set_title("Residual Distribution Comparison")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Cumulative distribution
for name, color in zip(methods_for_hist, colors):
    rs = residual_stats[name]
    axes[1].hist(np.abs(rs['residual'].flatten()), bins=100, alpha=0.7, cumulative=True, density=True,
                 label=f'{name}', color=color, histtype='step', linewidth=2)

axes[1].set_xlabel("|Residual|")
axes[1].set_ylabel("Cumulative Density")
axes[1].set_title("Cumulative Distribution of |Residual|")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Bar chart: PSNR vs Storage tradeoff
methods_for_bar = [name for name in comparison_results.keys() if "Per-Patch" in name or "Global" in name]
psnrs = [comparison_results[m]['psnr'] for m in methods_for_bar]
storage = [comparison_results[m]['storage_bytes'] / 1024 for m in methods_for_bar]  # KB

ax2 = axes[2]
x_pos = np.arange(len(methods_for_bar))
bars = ax2.bar(x_pos, psnrs, alpha=0.7, color='steelblue')
ax2.set_xticks(x_pos)
ax2.set_xticklabels([m.replace("Per-Patch Affine ", "").replace("Global Affine", "Global") for m in methods_for_bar], rotation=45, ha='right')
ax2.set_ylabel("PSNR (dB)")
ax2.set_title("PSNR vs Method")
ax2.grid(True, alpha=0.3, axis='y')

# Add storage labels on bars
for bar, stor in zip(bars, storage):
    ax2.annotate(f'{stor:.1f}KB', xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                 xytext=(0, 3), textcoords='offset points', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

# Print storage efficiency
print("\n💡 Storage Efficiency Analysis:")
print(f"   Global Affine: 24 bytes → {comparison_results['Global Affine']['psnr']:.2f} dB")
for ps in PATCH_SIZES:
    name = f"Per-Patch Affine ({ps})"
    stor = comparison_results[name]['storage_bytes']
    psnr = comparison_results[name]['psnr']
    improvement = psnr - comparison_results['Global Affine']['psnr']
    print(f"   Per-Patch ({ps:>3}): {stor:>6} bytes → {psnr:.2f} dB (Δ = {improvement:+.2f} dB vs global)")

### Test Per-Patch Affine with Overlap (Blending)

Using overlap between patches can reduce visible seams at patch boundaries.

In [ ]:
# Test with overlap for smoother blending
OVERLAP_AMOUNTS = [0, 16, 32]
PATCH_SIZE_TEST = 128

results_by_overlap = {}

for overlap in OVERLAP_AMOUNTS:
    print(f"\n{'='*60}")
    print(f"📐 Patch size={PATCH_SIZE_TEST}, Overlap={overlap}")
    print(f"{'='*60}")
    
    result = per_patch_affine_fitting(
        x_prev, x_target,
        patch_size=PATCH_SIZE_TEST,
        overlap=overlap,
        n_iterations=200,
        lr=0.02,
        device=device,
        verbose=True
    )
    
    results_by_overlap[overlap] = result
    m = result['metrics']
    print(f"   PSNR: {m['psnr']:.2f} dB, Improvement: +{m['improvement']:.2f} dB")
    print(f"   Patches: {result['n_patches']}")

# Compare
fig, axes = plt.subplots(2, len(OVERLAP_AMOUNTS) + 1, figsize=(5 * (len(OVERLAP_AMOUNTS) + 1), 10))

# Ground truth
axes[0, 0].imshow(x_target, vmin=0, vmax=1)
axes[0, 0].set_title("Ground Truth")
axes[0, 0].axis("off")
axes[1, 0].axis("off")

for col, overlap in enumerate(OVERLAP_AMOUNTS, 1):
    r = results_by_overlap[overlap]
    
    axes[0, col].imshow(r['warped_frame'], vmin=0, vmax=1)
    axes[0, col].set_title(f"Overlap={overlap}\nPSNR: {r['metrics']['psnr']:.2f} dB")
    axes[0, col].axis("off")
    
    diff = r['warped_frame'] - x_target
    vmax = max(np.abs(diff).max(), 1e-6)
    im = axes[1, col].imshow(diff, cmap='RdBu', vmin=-vmax, vmax=vmax)
    axes[1, col].set_title(f"Error (overlap={overlap})")
    axes[1, col].axis("off")
    plt.colorbar(im, ax=axes[1, col], fraction=0.046)

plt.suptitle(f"Per-Patch Affine Fitting: Effect of Overlap (patch_size={PATCH_SIZE_TEST})", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

# Summary
print("\n📊 Overlap Comparison:")
print(f"{'Overlap':>10} {'Num Patches':>12} {'PSNR (dB)':>12} {'Storage (KB)':>14}")
print("-"*50)
for overlap in OVERLAP_AMOUNTS:
    r = results_by_overlap[overlap]
    storage_kb = (r['n_patches'] * 6 * 4) / 1024
    print(f"{overlap:>10} {r['n_patches']:>12} {r['metrics']['psnr']:>12.2f} {storage_kb:>14.2f}")